In [8]:
import pandas as pd
import numpy as np
import time
import pickle

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score



df = pd.read_csv("/content/ml_ready_iot_data.csv")

# Clean column names
df.columns = df.columns.str.strip()

print("Data Loaded")
print(df.head())



df.replace("", np.nan, inplace=True)

df.fillna({
    "temperature": df["temperature"].mean(),
    "humidity": df["humidity"].mean(),
    "usage_hours": 0,
    "error_count": 0,
    "vibration": df["vibration"].median()
}, inplace=True)

numeric_cols = ["temperature", "humidity", "usage_hours", "error_count", "vibration"]
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')

df.drop_duplicates(inplace=True)

# Clip outliers
df["temperature"] = np.clip(df["temperature"], 20, 80)
df["humidity"] = np.clip(df["humidity"], 30, 100)
df["vibration"] = np.clip(df["vibration"], 0, 2)

df["device_status"] = df["device_status"].str.lower().str.strip()




# Avg temp
df["avg_temp"] = df.groupby("device_id")["temperature"].transform("mean")

# Temp change
df["temp_change"] = df.groupby("device_id")["temperature"].diff().fillna(0)

# Usage frequency
df["usage_frequency"] = df["usage_hours"] / df["usage_hours"].max()

# Health score
df["health_score"] = (
    100
    - df["temperature"] * 0.3
    - df["error_count"] * 5
    - df["vibration"] * 10
)



features = [
    "temperature", "humidity", "usage_hours",
    "error_count", "vibration", "avg_temp",
    "temp_change", "usage_frequency", "health_score"
]

X = df[features]
y = df["device_status"]



X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)



param_grid = {
    "n_estimators": [50, 100],
    "max_depth": [None, 5, 10]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=3,
    scoring='accuracy'
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_

print("\n Best Parameters:", grid.best_params_)



pred = best_model.predict(X_test)

print("\n Model Accuracy:", accuracy_score(y_test, pred))




Data Loaded
  device_id  temperature  humidity  usage_hours  error_count  vibration  \
0      D101         57.0        41           24            4       0.44   
1      D102         32.0        46           22            1       0.93   
2      D103         26.6        45            7            3       0.81   
3      D104         26.3        52           23            6       0.41   
4      D105         54.5        40            6            6       0.58   

   avg_temp  temp_change  usage_frequency  health_score health_status  \
0      57.0          0.0         1.000000         58.50      moderate   
1      32.0          0.0         0.916667         76.10          good   
2      26.6          0.0         0.291667         68.92      moderate   
3      26.3          0.0         0.958333         58.01      moderate   
4      54.5          0.0         0.250000         47.85      moderate   

   is_risky device_status  
0         0       warning  
1         0        normal  
2         0   

In [9]:

with open("rf_model.pkl", "wb") as f:
    pickle.dump(best_model, f)

print("\n Model Saved Successfully")


 Model Saved Successfully


In [10]:
for index, row in df.iterrows():
  input_data= row[features].values.reshape(1,-1)

  prediction=model.predict(input_data)[0]

  output={
      "device_id":row["device_id"],
      "status":prediction
  }

  print(output)


  time.sleep(1)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


{'device_id': 'D101', 'status': 'warning'}


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


{'device_id': 'D102', 'status': 'normal'}


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


...

{'device_id': 'D198', 'status': 'failure'}


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


{'device_id': 'D199', 'status': 'failure'}


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


{'device_id': 'D200', 'status': 'failure'}
